In [1]:
import torch
import timm

print(f"PyTorch version : {torch.__version__}")
print(f"timm version    : {timm.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"GPU count       : {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# Test if coatnet_0_rw_224 works on single GPU first
print("\nTesting single GPU forward pass...")
model = timm.create_model('coatnet_0_rw_224', pretrained=False, num_classes=1)
model = model.cuda()
x = torch.zeros(2, 3, 224, 224).cuda()

with torch.no_grad():
    out = model(x)

print(f"  Output shape: {out.shape}")
print("Single GPU test passed.")

PyTorch version : 2.10.0+cu128
timm version    : 1.0.25
CUDA available  : True
GPU count       : 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4

Testing single GPU forward pass...
  Output shape: torch.Size([2, 1])
Single GPU test passed.


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Configuration
SOURCE_ROOT = '/kaggle/input/datasets/awsaf49/artifact-dataset'
TARGET_ROOT = '/kaggle/working/custom_artifact_v3'

TRAIN_SIZE  = 50000
VAL_SIZE    = 5000
TEST_SIZE   = 10000
RANDOM_SEED = 42

EXCLUDED_GENERATORS = {'stylegan2', 'diffusion_gan', 'generative_inpainting'}

# Stage 1: Metadata Aggregation
print("Stage 1: Aggregating metadata from all generator folders...")

all_metadata = []
skipped      = []

for folder_name in sorted(os.listdir(SOURCE_ROOT)):
    folder_path = os.path.join(SOURCE_ROOT, folder_name)

    if not os.path.isdir(folder_path):
        continue

    csv_path = os.path.join(folder_path, 'metadata.csv')
    if not os.path.exists(csv_path):
        skipped.append(folder_name)
        continue

    df = pd.read_csv(csv_path)

    if 'image_path' not in df.columns or 'target' not in df.columns:
        skipped.append(folder_name)
        continue

    correct_base = None
    for img_path in df['image_path'].head(10):
        clean = str(img_path).lstrip('/')
        if os.path.exists(os.path.join(folder_path, clean)):
            correct_base = folder_path
            break
        elif os.path.exists(os.path.join(SOURCE_ROOT, clean)):
            correct_base = SOURCE_ROOT
            break

    if correct_base is None:
        skipped.append(folder_name)
        print(f"  [WARN] Could not resolve paths for: {folder_name}")
        continue

    df['absolute_path'] = df['image_path'].apply(
        lambda x: os.path.join(correct_base, str(x).lstrip('/'))
    )
    df['generator_id'] = folder_name
    df['is_fake']      = (df['target'] != 0).astype(int)
    all_metadata.append(df[['absolute_path', 'generator_id', 'is_fake']])

master_df = pd.concat(all_metadata, ignore_index=True)
print(f"  Total images before filtering : {len(master_df):,}")

# Stage 2: Exclude Hard Generators
print("\nStage 2: Excluding hard generators...")

before_fake = (master_df['is_fake'] == 1).sum()

master_df = master_df[
    ~((master_df['is_fake'] == 1) &
      (master_df['generator_id'].isin(EXCLUDED_GENERATORS)))
].copy()

after_fake = (master_df['is_fake'] == 1).sum()

print(f"  Excluded generators : {EXCLUDED_GENERATORS}")
print(f"  Fake images before  : {before_fake:,}")
print(f"  Fake images after   : {after_fake:,}")
print(f"  Removed             : {before_fake - after_fake:,}")
print(f"  Remaining generators: "
      f"{master_df[master_df['is_fake']==1]['generator_id'].nunique()}")

# Stage 3: Stratified Sampling per Generator
print("\nStage 3: Stratified sampling per generator...")

fake_df = master_df[master_df['is_fake'] == 1].copy()
real_df = master_df[master_df['is_fake'] == 0].copy()

total_needed   = TRAIN_SIZE + VAL_SIZE + TEST_SIZE
generators     = fake_df['generator_id'].unique()
quota_per_gen  = total_needed // len(generators)
remainder      = total_needed  % len(generators)

print(f"  Fake generators : {len(generators)}")
print(f"  Quota per gen   : {quota_per_gen} (+{remainder} remainder)")

sampled_fake_parts = []
for gen in generators:
    gen_df = fake_df[fake_df['generator_id'] == gen]
    n      = min(quota_per_gen, len(gen_df))
    sampled_fake_parts.append(gen_df.sample(n=n, random_state=RANDOM_SEED))

fake_subset = pd.concat(sampled_fake_parts, ignore_index=True)

shortfall = total_needed - len(fake_subset)
if shortfall > 0:
    remaining = fake_df.drop(fake_subset.index, errors='ignore')
    fill      = remaining.sample(
        n=min(shortfall, len(remaining)), random_state=RANDOM_SEED)
    fake_subset = pd.concat([fake_subset, fill], ignore_index=True)

real_subset = real_df.sample(n=total_needed, random_state=RANDOM_SEED)

print(f"  Fake subset size : {len(fake_subset):,}")
print(f"  Real subset size : {len(real_subset):,}")

# Stage 4: Train / Val / Test Split
print("\nStage 4: Splitting into train/val/test...")

def stratified_split(df, train_n, val_n, test_n, seed):
    train, temp = train_test_split(df, train_size=train_n,
                                   random_state=seed, shuffle=True)
    val, test   = train_test_split(temp, train_size=val_n,
                                   random_state=seed, shuffle=True)
    return train, val, test

real_train, real_val, real_test = stratified_split(
    real_subset, TRAIN_SIZE, VAL_SIZE, TEST_SIZE, RANDOM_SEED)
fake_train, fake_val, fake_test = stratified_split(
    fake_subset, TRAIN_SIZE, VAL_SIZE, TEST_SIZE, RANDOM_SEED)

train_df = pd.concat([real_train, fake_train]).sample(
    frac=1, random_state=RANDOM_SEED)
val_df   = pd.concat([real_val,   fake_val  ]).sample(
    frac=1, random_state=RANDOM_SEED)
test_df  = pd.concat([real_test,  fake_test ]).sample(
    frac=1, random_state=RANDOM_SEED)

# Stage 5: Export and Verify
print("\nStage 5: Exporting CSV manifests...")
os.makedirs(TARGET_ROOT, exist_ok=True)
train_df.to_csv(os.path.join(TARGET_ROOT, 'train.csv'), index=False)
val_df.to_csv(  os.path.join(TARGET_ROOT, 'val.csv'),   index=False)
test_df.to_csv( os.path.join(TARGET_ROOT, 'test.csv'),  index=False)

print("\nStage 6: Verification...")
for split_name, df in [('TRAIN', train_df), ('VAL', val_df), ('TEST', test_df)]:
    real_c = (df['is_fake'] == 0).sum()
    fake_c = (df['is_fake'] == 1).sum()
    gens   = df[df['is_fake'] == 1]['generator_id'].nunique()
    print(f"  {split_name:<6} | Total: {len(df):>7,} | "
          f"Real: {real_c:>6,} | Fake: {fake_c:>6,} | Generators: {gens}")

print("\nManifest build complete.")

Stage 1: Aggregating metadata from all generator folders...
  Total images before filtering : 2,496,738

Stage 2: Excluding hard generators...
  Excluded generators : {'generative_inpainting', 'stylegan2', 'diffusion_gan'}
  Fake images before  : 1,531,749
  Fake images after   : 494,242
  Removed             : 1,037,507
  Remaining generators: 22

Stage 3: Stratified sampling per generator...
  Fake generators : 22
  Quota per gen   : 2954 (+12 remainder)
  Fake subset size : 65,000
  Real subset size : 65,000

Stage 4: Splitting into train/val/test...

Stage 5: Exporting CSV manifests...

Stage 6: Verification...
  TRAIN  | Total: 100,000 | Real: 50,000 | Fake: 50,000 | Generators: 22
  VAL    | Total:  10,000 | Real:  5,000 | Fake:  5,000 | Generators: 22
  TEST   | Total:  20,000 | Real: 10,000 | Fake: 10,000 | Generators: 22

Manifest build complete.


In [ ]:
import cv2
import io
import numpy as np
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

# Disable OpenCV internal threading to prevent collision with DataLoader workers
cv2.setNumThreads(0)

# ELA and FFT Feature Extraction Utilities

def compute_ela(image_rgb: np.ndarray, quality: int = 90) -> np.ndarray:
    """
    Error Level Analysis: compress image at given quality and compute
    the absolute difference. High-ELA regions indicate potential manipulation.
    Returns a single-channel float32 map normalized to [0, 1].
    """
    pil_img = Image.fromarray(image_rgb)
    buffer  = io.BytesIO()
    pil_img.save(buffer, format='JPEG', quality=quality)
    buffer.seek(0)
    compressed = np.array(Image.open(buffer).convert('RGB')).astype(np.float32)
    original   = image_rgb.astype(np.float32)

    ela_map = np.abs(original - compressed).mean(axis=2)   # (H, W)
    ela_max = ela_map.max()
    if ela_max > 0:
        ela_map /= ela_max
    return ela_map.astype(np.float32)


def compute_fft_magnitude(image_rgb: np.ndarray) -> np.ndarray:
    """
    Compute log-scaled FFT magnitude spectrum averaged across RGB channels.
    Synthetic images leave periodic artifacts in the frequency domain.
    Returns a single-channel float32 map normalized to [0, 1].
    """
    gray     = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    fft      = np.fft.fft2(gray)
    fft_shift= np.fft.fftshift(fft)
    magnitude= np.log1p(np.abs(fft_shift))

    mag_max  = magnitude.max()
    if mag_max > 0:
        magnitude /= mag_max
    return magnitude.astype(np.float32)


# 5-Channel Dataset

class DeepfakeDataset5Ch(Dataset):
    """
    Loads images as 5-channel tensors: [R, G, B, ELA, FFT_magnitude].
    Training images are resized to 256x256 for random crop augmentation on GPU.
    Validation/Test images are resized directly to 224x224.
    """

    def __init__(self, csv_file: str, is_train: bool = True):
        df           = pd.read_csv(csv_file)
        self.paths   = df['absolute_path'].tolist()
        self.labels  = df['is_fake'].astype(float).tolist()
        self.is_train= is_train
        self.load_size = (256, 256) if is_train else (224, 224)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        label    = self.labels[idx]

        image_bgr = cv2.imread(img_path)

        # Fallback for corrupted files — avoids training crashes
        if image_bgr is None:
            dummy = torch.zeros((5, *self.load_size), dtype=torch.uint8)
            return dummy, torch.tensor([label], dtype=torch.float32)

        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        image_rgb = cv2.resize(image_rgb, self.load_size)

        #  Compute forensic channels 
        ela_map = compute_ela(image_rgb)                        # (H, W) float32
        fft_map = compute_fft_magnitude(image_rgb)              # (H, W) float32

        # Scale forensic channels to uint8 range for consistent GPU normalization
        ela_u8  = (ela_map * 255).astype(np.uint8)
        fft_u8  = (fft_map * 255).astype(np.uint8)

        #  Build 5-channel array: (H, W, 5) 
        ch5 = np.concatenate([
            image_rgb,                                          # (H, W, 3)
            ela_u8[..., np.newaxis],                            # (H, W, 1)
            fft_u8[..., np.newaxis]                             # (H, W, 1)
        ], axis=2)

        # HWC -> CHW, contiguous memory layout for fast GPU transfer
        image_tensor = torch.from_numpy(ch5).permute(2, 0, 1).contiguous()

        return image_tensor, torch.tensor([label], dtype=torch.float32 )


# Sanity Check
print("Running dataset sanity check...")

_ds    = DeepfakeDataset5Ch(
    csv_file=f'{TARGET_ROOT}/train.csv', is_train=True)
_img, _lbl = _ds[0]

print(f"  Dataset length    : {len(_ds):,}")
print(f"  Tensor shape      : {_img.shape}")
print(f"  Tensor dtype      : {_img.dtype}")
print(f"  Label value       : {_lbl.item()}")
print(f"  Channel min/max   : {_img.min().item()} / {_img.max().item()}")
print("Sanity check passed.")

Running dataset sanity check...
  Dataset length    : 100,000
  Tensor shape      : torch.Size([5, 256, 256])
  Tensor dtype      : torch.uint8
  Label value       : 1.0
  Channel min/max   : 0 / 255
Sanity check passed.


In [ ]:
import timm
import torch
import torch.nn as nn

# CoAtNet-Tiny: Single GPU + Gradient Accumulation Strategy

def build_coatnet_5ch(num_classes: int = 1, pretrained: bool = True) -> nn.Module:
    """
    Loads CoAtNet-Tiny pretrained on ImageNet-1k via timm.
    Adapts first conv layer to accept 5-channel input while
    preserving pretrained RGB weights for channels 0-2.
    Forensic channels (3,4) use He initialization.
    """
    model = timm.create_model(
        'coatnet_0_rw_224',
        pretrained=pretrained,
        num_classes=num_classes
    )

    original_conv = model.stem.conv1
    out_ch      = original_conv.out_channels
    kernel      = original_conv.kernel_size
    stride      = original_conv.stride
    padding     = original_conv.padding
    bias_exists = original_conv.bias is not None

    new_conv = nn.Conv2d(
        in_channels  = 5,
        out_channels = out_ch,
        kernel_size  = kernel,
        stride       = stride,
        padding      = padding,
        bias         = bias_exists
    )

    with torch.no_grad():
        new_conv.weight[:, :3, :, :] = original_conv.weight.clone()
        nn.init.kaiming_normal_(
            new_conv.weight[:, 3:, :, :],
            mode='fan_out',
            nonlinearity='relu'
        )
        if bias_exists:
            new_conv.bias.data = original_conv.bias.clone()

    model.stem.conv1 = new_conv
    return model


# Device Setup — Single GPU, Gradient Accumulation for
# effective large batch across both T4s is handled in
# the training loop via ACCUM_STEPS

PHYSICAL_BATCH  = 128    # fits comfortably in single T4 16GB with 5-ch input
ACCUM_STEPS     = 4      # effective batch = 128 * 4 = 512
EPOCHS_PHASE_A  = 5
EPOCHS_PHASE_B  = 5
EPOCHS_PHASE_C  = 10
TOTAL_EPOCHS    = EPOCHS_PHASE_A + EPOCHS_PHASE_B + EPOCHS_PHASE_C

device = torch.device('cuda:0')

print("Building CoAtNet-Tiny with 5-channel input...")
model = build_coatnet_5ch(num_classes=1, pretrained=True)
model = model.to(device)

base         = model
total_params = sum(p.numel() for p in base.parameters())
trainable    = sum(p.numel() for p in base.parameters() if p.requires_grad)

print(f"\n  Model             : coatnet_0_rw_224 (5-ch adapted)")
print(f"  Device            : {device} ({torch.cuda.get_device_name(0)})")
print(f"  Total params      : {total_params:,}")
print(f"  Trainable params  : {trainable:,}")
print(f"  Physical batch    : {PHYSICAL_BATCH}")
print(f"  Accumulation steps: {ACCUM_STEPS}")
print(f"  Effective batch   : {PHYSICAL_BATCH * ACCUM_STEPS}")
print(f"  Total epochs      : {TOTAL_EPOCHS}")

# Forward Pass Sanity Check
print("\nRunning forward pass sanity check...")
dummy_input = torch.zeros(2, 5, 224, 224, device=device)

with torch.no_grad():
    dummy_out = model(dummy_input)

print(f"  Input shape       : {dummy_input.shape}")
print(f"  Output shape      : {dummy_out.shape}")
print(f"  Output range      : [{dummy_out.min().item():.4f}, {dummy_out.max().item():.4f}]")

# Memory report after model load
allocated = torch.cuda.memory_allocated(0) / 1e9
reserved  = torch.cuda.memory_reserved(0)  / 1e9
print(f"\n  GPU memory allocated : {allocated:.2f} GB")
print(f"  GPU memory reserved  : {reserved:.2f} GB")
print("Architecture setup complete.")

Building CoAtNet-Tiny with 5-channel input...


model.safetensors:   0%|          | 0.00/110M [00:00<?, ?B/s]


  Model             : coatnet_0_rw_224 (5-ch adapted)
  Device            : cuda:0 (Tesla T4)
  Total params      : 26,667,907
  Trainable params  : 26,667,907
  Physical batch    : 128
  Accumulation steps: 4
  Effective batch   : 512
  Total epochs      : 20

Running forward pass sanity check...
  Input shape       : torch.Size([2, 5, 224, 224])
  Output shape      : torch.Size([2, 1])
  Output range      : [-0.7635, -0.7635]

  GPU memory allocated : 0.12 GB
  GPU memory reserved  : 0.19 GB
Architecture setup complete.


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np

cv2.setNumThreads(0)

# Focal Loss

class FocalLoss(nn.Module):
    """
    Focal Loss for binary classification.
    Reduces loss contribution from easy examples, focusing
    training on hard misclassified samples.
    FL(p) = -alpha * (1 - p)^gamma * log(p)
    """
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor,
                targets: torch.Tensor) -> torch.Tensor:
        bce_loss = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )
        probs      = torch.sigmoid(logits)
        # For positive class: p_t = probs, for negative: p_t = 1 - probs
        p_t        = probs * targets + (1 - probs) * (1 - targets)
        alpha_t    = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_w    = alpha_t * (1 - p_t) ** self.gamma
        return (focal_w * bce_loss).mean()


# Hard Negative Mining — Sample Weights Update

def compute_sample_weights(model: nn.Module,
                            csv_path: str,
                            device: torch.device,
                            top_hard_fraction: float = 0.20,
                            hard_weight_multiplier: float = 3.0,
                            batch_size: int = 256) -> np.ndarray:
    """
    Performs a no-grad forward pass over the training set to compute
    per-sample loss. The top hard_fraction of samples by loss value
    receive a multiplied sampling weight for the next epoch.
    Returns a numpy array of per-sample weights for WeightedRandomSampler.
    """
    model.eval()

    mean = torch.tensor([0.485, 0.456, 0.406, 0.5, 0.5],
                        device=device).view(1, 5, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225, 0.5, 0.5],
                        device=device).view(1, 5, 1, 1)

    # Temporary loader — no shuffle, full dataset
    temp_ds     = DeepfakeDataset5Ch(csv_path, is_train=False)
    temp_loader = DataLoader(temp_ds, batch_size=batch_size,
                             shuffle=False, num_workers=4,
                             pin_memory=True, prefetch_factor=2,
                             persistent_workers=True)

    per_sample_loss = []
    criterion       = nn.BCEWithLogitsLoss(reduction='none')

    with torch.no_grad():
        for images, labels in temp_loader:
            images = images.to(device, non_blocking=True).float() / 255.0
            images = (images - mean) / std
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss   = criterion(logits, labels)

            per_sample_loss.extend(loss.squeeze().cpu().numpy())

    per_sample_loss = np.array(per_sample_loss)

    # Assign base weight 1.0, hard samples get multiplied weight
    weights          = np.ones(len(per_sample_loss), dtype=np.float32)
    threshold        = np.quantile(per_sample_loss, 1.0 - top_hard_fraction)
    weights[per_sample_loss >= threshold] = hard_weight_multiplier

    model.train()
    return weights


# DataLoaders Factory

def create_dataloaders(train_csv: str, val_csv: str, test_csv: str,
                       physical_batch: int,
                       sample_weights: np.ndarray = None):
    """
    Creates train/val/test DataLoaders.
    If sample_weights is provided, uses WeightedRandomSampler for
    hard negative mining; otherwise uses standard random shuffle.
    """
    train_ds = DeepfakeDataset5Ch(train_csv, is_train=True)
    val_ds   = DeepfakeDataset5Ch(val_csv,   is_train=False)
    test_ds  = DeepfakeDataset5Ch(test_csv,  is_train=False)

    if sample_weights is not None:
        sampler      = WeightedRandomSampler(
            weights     = torch.from_numpy(sample_weights),
            num_samples = len(sample_weights),
            replacement = True
        )
        train_loader = DataLoader(
            train_ds, batch_size=physical_batch,
            sampler=sampler, num_workers=4,
            pin_memory=True, prefetch_factor=4,
            persistent_workers=True
        )
    else:
        train_loader = DataLoader(
            train_ds, batch_size=physical_batch,
            shuffle=True, num_workers=4,
            pin_memory=True, prefetch_factor=4,
            persistent_workers=True, drop_last=True
        )

    val_loader  = DataLoader(
        val_ds, batch_size=physical_batch * 2,
        shuffle=False, num_workers=4,
        pin_memory=True, prefetch_factor=4,
        persistent_workers=True
    )
    test_loader = DataLoader(
        test_ds, batch_size=physical_batch * 2,
        shuffle=False, num_workers=4,
        pin_memory=True, prefetch_factor=4,
        persistent_workers=True
    )
    return train_loader, val_loader, test_loader


# Gradual Unfreezing Controller

def apply_gradual_unfreezing(model: nn.Module, epoch: int,
                              phase_a_end: int,
                              phase_b_end: int) -> float:
    """
    Phase A: stem.conv1 + norm + head only (stabilize new layers)
    Phase B: + stages[3] (deepest attention stage)
    Phase C: all parameters
    """
    # Freeze everything first
    for param in model.parameters():
        param.requires_grad = False

    if epoch <= phase_a_end:
        for param in model.stem.conv1.parameters():
            param.requires_grad = True
        for param in model.norm.parameters():
            param.requires_grad = True
        for param in model.head.parameters():
            param.requires_grad = True
        lr    = 1e-3
        phase = 'A'

    elif epoch <= phase_b_end:
        for param in model.stem.conv1.parameters():
            param.requires_grad = True
        for param in model.norm.parameters():
            param.requires_grad = True
        for param in model.head.parameters():
            param.requires_grad = True
        for param in model.stages[-1].parameters():
            param.requires_grad = True
        lr    = 1e-4
        phase = 'B'

    else:
        for param in model.parameters():
            param.requires_grad = True
        lr    = 1e-5
        phase = 'C'

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  [Phase {phase}] Trainable: {trainable:,} / {total:,} "
          f"({100*trainable/total:.1f}%) | LR: {lr}")
    return lr



# Sanity Checks
print("Verifying Focal Loss...")
_logits  = torch.randn(4, 1)
_targets = torch.randint(0, 2, (4, 1)).float()
_fl      = FocalLoss(alpha=0.25, gamma=2.0)
_loss    = _fl(_logits, _targets)
print(f"  Focal Loss output : {_loss.item():.4f}")

print("\nVerifying DataLoaders...")
_train_loader, _val_loader, _test_loader = create_dataloaders(
    train_csv = f'{TARGET_ROOT}/train.csv',
    val_csv   = f'{TARGET_ROOT}/val.csv',
    test_csv  = f'{TARGET_ROOT}/test.csv',
    physical_batch = PHYSICAL_BATCH
)
_imgs, _lbls = next(iter(_train_loader))
print(f"  Train batch shape : {_imgs.shape}")
print(f"  Label dtype       : {_lbls.dtype}")

print("\nVerifying Gradual Unfreezing (Phase A simulation)...")
apply_gradual_unfreezing(model, epoch=1,
                         phase_a_end=EPOCHS_PHASE_A,
                         phase_b_end=EPOCHS_PHASE_A + EPOCHS_PHASE_B)

print("\nAll utilities ready.")

Verifying Focal Loss...
  Focal Loss output : 0.1812

Verifying DataLoaders...
  Train batch shape : torch.Size([128, 5, 256, 256])
  Label dtype       : torch.float32

Verifying Gradual Unfreezing (Phase A simulation)...
  [Phase A] Trainable: 3,745 / 26,667,907 (0.0%) | LR: 0.001

All utilities ready.


In [ ]:
import os
import time
import json
import torch
import torch.optim as optim
from sklearn.metrics import accuracy_score, roc_auc_score, precision_recall_fscore_support

# GPU Augmentation — applied on-device to eliminate CPU bottleneck

def gpu_augment_train(images: torch.Tensor, device: torch.device) -> torch.Tensor:
    """
    Performs all augmentation on GPU after uint8->float conversion.
    Channels 0-2: ImageNet normalization
    Channels 3-4: ELA/FFT normalized to zero mean unit variance
    Random crop 256->224 and horizontal flip are applied batch-wise.
    """
    mean = torch.tensor([0.485, 0.456, 0.406, 0.5, 0.5],
                        device=device).view(1, 5, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225, 0.5, 0.5],
                        device=device).view(1, 5, 1, 1)

    images = images.float() / 255.0
    images = (images - mean) / std

    # Random crop: 256 -> 224
    h_start = torch.randint(0, 33, (1,)).item()
    w_start = torch.randint(0, 33, (1,)).item()
    images  = images[:, :, h_start:h_start+224, w_start:w_start+224]

    # Random horizontal flip
    if torch.rand(1).item() > 0.5:
        images = torch.flip(images, dims=[3])

    return images


def gpu_normalize_val(images: torch.Tensor, device: torch.device) -> torch.Tensor:
    """Normalize only, no augmentation for val/test."""
    mean = torch.tensor([0.485, 0.456, 0.406, 0.5, 0.5],
                        device=device).view(1, 5, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225, 0.5, 0.5],
                        device=device).view(1, 5, 1, 1)
    images = images.float() / 255.0
    return (images - mean) / std


# Training Loop with Gradual Unfreezing + Grad Accumulation

def train_and_validate(model, train_csv, val_csv, test_csv,
                       device, save_dir='artifacts_v2'):

    os.makedirs(save_dir, exist_ok=True)
    best_model_path = os.path.join(save_dir, 'best_coatnet_5ch.pth')
    history_path    = os.path.join(save_dir, 'training_history.json')

    criterion  = FocalLoss(alpha=0.25, gamma=2.0)
    scaler     = torch.amp.GradScaler('cuda')
    history    = {k: [] for k in ['train_loss', 'val_loss',
                                   'val_auc', 'val_f1', 'val_acc']}
    best_auc   = 0.0
    sample_weights = None

    for epoch in range(1, TOTAL_EPOCHS + 1):
        epoch_start = time.time()

        # --- Gradual Unfreezing: update trainable params and LR ---
        current_lr = apply_gradual_unfreezing(
            model, epoch,
            phase_a_end = EPOCHS_PHASE_A,
            phase_b_end = EPOCHS_PHASE_A + EPOCHS_PHASE_B
        )

        # --- Hard Negative Mining: recompute weights after Phase A ---
        if epoch == EPOCHS_PHASE_A + 1:
            print("  Computing hard negative weights...")
            sample_weights = compute_sample_weights(
                model, train_csv, device,
                top_hard_fraction    = 0.20,
                hard_weight_multiplier = 3.0,
                batch_size           = PHYSICAL_BATCH * 2
            )
            print(f"  Hard negatives identified: "
                  f"{(sample_weights > 1.0).sum():,} / {len(sample_weights):,}")

        # --- Rebuild DataLoaders each epoch only when weights change ---
        # Weights change once (at phase transition), stable otherwise
        if epoch == 1 or epoch == EPOCHS_PHASE_A + 1:
            train_loader, val_loader, _ = create_dataloaders(
                train_csv, val_csv, test_csv,
                PHYSICAL_BATCH, sample_weights
            )

        # --- Rebuild optimizer with correct LR for this phase ---
        trainable_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = optim.AdamW(trainable_params, lr=current_lr,
                                weight_decay=0.01)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS_PHASE_A, eta_min=current_lr * 0.1
        )

        # -------------------------------------------------------
        # Training Phase
        # -------------------------------------------------------
        model.train()
        running_loss  = 0.0
        optimizer.zero_grad(set_to_none=True)

        for step, (images, labels) in enumerate(train_loader):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            images = gpu_augment_train(images, device)

            with torch.amp.autocast('cuda'):
                outputs = model(images)
                # Normalize loss by accumulation steps
                loss    = criterion(outputs, labels) / ACCUM_STEPS

            scaler.scale(loss).backward()

            # Gradient update every ACCUM_STEPS steps
            if (step + 1) % ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            running_loss += loss.item() * ACCUM_STEPS * images.size(0)

        # Handle remaining steps not divisible by ACCUM_STEPS
        if (len(train_loader)) % ACCUM_STEPS != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        scheduler.step()
        epoch_train_loss = running_loss / len(train_loader.dataset)

        # -------------------------------------------------------
        # Validation Phase
        # -------------------------------------------------------
        model.eval()
        val_loss                        = 0.0
        all_labels, all_probs, all_preds = [], [], []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                images = gpu_normalize_val(images, device)

                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss    = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                probs     = torch.sigmoid(outputs)
                preds     = (probs > 0.5).float()

                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())

        epoch_val_loss = val_loss / len(val_loader.dataset)
        acc   = accuracy_score(all_labels, all_preds)
        _, _, f1, _ = precision_recall_fscore_support(
            all_labels, all_preds, average='binary', zero_division=0)
        auc   = roc_auc_score(all_labels, all_probs)

        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(float(acc))
        history['val_f1'].append(float(f1))
        history['val_auc'].append(float(auc))

        elapsed = time.time() - epoch_start
        print(f"  Epoch {epoch:02d}/{TOTAL_EPOCHS} [{elapsed:.0f}s] "
              f"Train Loss: {epoch_train_loss:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f} | "
              f"AUC: {auc:.4f} | F1: {f1:.4f} | Acc: {acc:.4f}")

        # Save best model
        if auc > best_auc:
            best_auc = auc
            torch.save(model.state_dict(), best_model_path)
            print(f"  Best model saved (AUC: {best_auc:.4f})")

        # Periodic memory cleanup to prevent fragmentation
        torch.cuda.empty_cache()

    with open(history_path, 'w') as f:
        json.dump(history, f, indent=2)

    print(f"\nTraining complete. Best Val AUC: {best_auc:.4f}")
    return best_model_path


# Sanity Check: single batch forward+backward
print("Running single-batch sanity check before full training...")

apply_gradual_unfreezing(model, epoch=1,
                         phase_a_end=EPOCHS_PHASE_A,
                         phase_b_end=EPOCHS_PHASE_A + EPOCHS_PHASE_B)

_criterion = FocalLoss(alpha=0.25, gamma=2.0)
_scaler    = torch.amp.GradScaler('cuda')
_params    = [p for p in model.parameters() if p.requires_grad]
_optimizer = optim.AdamW(_params, lr=1e-3, weight_decay=0.01)

_ds     = DeepfakeDataset5Ch(f'{TARGET_ROOT}/train.csv', is_train=True)
_loader = DataLoader(_ds, batch_size=PHYSICAL_BATCH, shuffle=True,
                     num_workers=2, pin_memory=True)
_imgs, _lbls = next(iter(_loader))
_imgs = _imgs.to(device)
_lbls = _lbls.to(device)
_imgs = gpu_augment_train(_imgs, device)

_optimizer.zero_grad(set_to_none=True)
with torch.amp.autocast('cuda'):
    _out  = model(_imgs)
    _loss = _criterion(_out, _lbls)

_scaler.scale(_loss).backward()
_scaler.step(_optimizer)
_scaler.update()

allocated = torch.cuda.memory_allocated(0) / 1e9
reserved  = torch.cuda.memory_reserved(0)  / 1e9

print(f"  Loss value        : {_loss.item():.4f}")
print(f"  Output shape      : {_out.shape}")
print(f"  GPU allocated     : {allocated:.2f} GB")
print(f"  GPU reserved      : {reserved:.2f} GB")
print("Sanity check passed. Ready for full training.")

Running single-batch sanity check before full training...
  [Phase A] Trainable: 3,745 / 26,667,907 (0.0%) | LR: 0.001
  Loss value        : 0.1075
  Output shape      : torch.Size([128, 1])
  GPU allocated     : 0.26 GB
  GPU reserved      : 12.37 GB
Sanity check passed. Ready for full training.


In [ ]:
import json
from sklearn.metrics import (accuracy_score, roc_auc_score,
                              precision_recall_fscore_support,
                              confusion_matrix)

# Launch Full Training
print("=" * 60)
print("Starting full training pipeline")
print(f"  Epochs     : {TOTAL_EPOCHS} "
      f"(A:{EPOCHS_PHASE_A} / B:{EPOCHS_PHASE_B} / C:{EPOCHS_PHASE_C})")
print(f"  Batch      : {PHYSICAL_BATCH} x {ACCUM_STEPS} = "
      f"{PHYSICAL_BATCH * ACCUM_STEPS} effective")
print(f"  Train size : 100,000 | Val: 10,000 | Test: 20,000")
print("=" * 60)

SAVE_DIR = '/kaggle/working/artifacts_v2'

best_model_path = train_and_validate(
    model     = model,
    train_csv = f'{TARGET_ROOT}/train.csv',
    val_csv   = f'{TARGET_ROOT}/val.csv',
    test_csv  = f'{TARGET_ROOT}/test.csv',
    device    = device,
    save_dir  = SAVE_DIR
)

# Final Evaluation on Test Set

def evaluate_on_test(model, test_csv, best_model_path, device, save_dir):
    print("\n" + "=" * 60)
    print("Final Evaluation on Test Set")
    print("=" * 60)

    # Restore best weights
    model.load_state_dict(
        torch.load(best_model_path, map_location=device, weights_only=True)
    )
    model.eval()

    _, _, test_loader = create_dataloaders(
        train_csv = f'{TARGET_ROOT}/train.csv',
        val_csv   = f'{TARGET_ROOT}/val.csv',
        test_csv  = test_csv,
        physical_batch  = PHYSICAL_BATCH * 2,
        sample_weights  = None
    )

    all_labels, all_probs, all_preds = [], [], []
    start = time.time()

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device, non_blocking=True)
            images = gpu_normalize_val(images, device)

            with torch.amp.autocast('cuda'):
                outputs = model(images)

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    elapsed   = time.time() - start
    acc       = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary', zero_division=0)
    auc       = roc_auc_score(all_labels, all_probs)
    cm        = confusion_matrix(all_labels, all_preds)

    test_metrics = {
        "test_accuracy"   : round(float(acc),       4),
        "test_precision"  : round(float(precision),  4),
        "test_recall"     : round(float(recall),     4),
        "test_f1_score"   : round(float(f1),         4),
        "test_auc_roc"    : round(float(auc),        4),
        "confusion_matrix": cm.tolist()
    }

    with open(os.path.join(save_dir, 'test_metrics.json'), 'w') as f:
        json.dump(test_metrics, f, indent=4)

    print(f"  Evaluation time : {elapsed:.0f}s")
    print(f"  AUC-ROC         : {auc:.4f}")
    print(f"  F1-Score        : {f1:.4f}")
    print(f"  Accuracy        : {acc:.4f}")
    print(f"  Precision       : {precision:.4f}")
    print(f"  Recall          : {recall:.4f}")
    print(f"\n  Confusion Matrix:")
    print(f"    TN: {cm[0][0]:>6,}  FP: {cm[0][1]:>6,}")
    print(f"    FN: {cm[1][0]:>6,}  TP: {cm[1][1]:>6,}")

    return test_metrics


test_metrics = evaluate_on_test(
    model           = model,
    test_csv        = f'{TARGET_ROOT}/test.csv',
    best_model_path = best_model_path,
    device          = device,
    save_dir        = SAVE_DIR
)

Starting full training pipeline
  Epochs     : 20 (A:5 / B:5 / C:10)
  Batch      : 128 x 4 = 512 effective
  Train size : 100,000 | Val: 10,000 | Test: 20,000
  [Phase A] Trainable: 3,745 / 26,667,907 (0.0%) | LR: 0.001
  Epoch 01/20 [644s] Train Loss: 0.0600 | Val Loss: 0.0564 | AUC: 0.8166 | F1: 0.6240 | Acc: 0.7014
  Best model saved (AUC: 0.8166)
  [Phase A] Trainable: 3,745 / 26,667,907 (0.0%) | LR: 0.001
  Epoch 02/20 [572s] Train Loss: 0.0550 | Val Loss: 0.0532 | AUC: 0.8416 | F1: 0.6737 | Acc: 0.7293
  Best model saved (AUC: 0.8416)
  [Phase A] Trainable: 3,745 / 26,667,907 (0.0%) | LR: 0.001
  Epoch 03/20 [572s] Train Loss: 0.0534 | Val Loss: 0.0528 | AUC: 0.8436 | F1: 0.6718 | Acc: 0.7282
  Best model saved (AUC: 0.8436)
  [Phase A] Trainable: 3,745 / 26,667,907 (0.0%) | LR: 0.001
  Epoch 04/20 [573s] Train Loss: 0.0524 | Val Loss: 0.0517 | AUC: 0.8495 | F1: 0.6456 | Acc: 0.7184
  Best model saved (AUC: 0.8495)
  [Phase A] Trainable: 3,745 / 26,667,907 (0.0%) | LR: 0.001
  E